In [1]:
!pip install -q fastapi uvicorn python-multipart pyngrok pypdf langchain langchain-core langchain-community langchain-text-splitters langchain-google-genai faiss-cpu pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 794.0 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.7/81.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 11.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not ins

In [3]:
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = s.get_secret("GOOGLE_API_KEY")
NGROK_TOKEN = s.get_secret("NGROK_TOKEN")
os.environ["API_PASSWORD"] = s.get_secret("API_PASSWORD")
try:
    os.environ["CHAT_MODEL"] = s.get_secret("CHAT_MODEL")
except Exception:
    os.environ["CHAT_MODEL"] = "gemini-flash-lite-latest"   # change if this model is retired
print("Keys loaded. Chat model:", os.environ["CHAT_MODEL"])

Keys loaded. Chat model: gemini-flash-lite-latest


In [4]:
import os
WORKDIR = "/kaggle/working/interview_api"
os.makedirs(WORKDIR + "/data", exist_ok=True)
os.chdir(WORKDIR)
print("Working in", os.getcwd())

Working in /kaggle/working/interview_api


In [5]:
%%writefile lab.py
import os
import time
import uuid
from functools import lru_cache
from pathlib import Path
from typing import List

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# ---- Config ----
CHAT_MODEL = os.getenv("CHAT_MODEL", "gemini-flash-lite-latest")
EMBED_MODEL = os.getenv("EMBED_MODEL", "models/gemini-embedding-001")
DATA_DIR = Path(__file__).parent / "data"


# ---- Output parser schemas ----
class QuestionList(BaseModel):
    questions: List[str] = Field(description="Interview questions, one per item")


class Evaluation(BaseModel):
    score: int = Field(description="Score from 0 to 10", ge=0, le=10)
    missing_points: List[str] = Field(description="Important points the answer missed")
    suggestions: List[str] = Field(description="Concrete tips to improve the answer")
    feedback: str = Field(description="One short paragraph of overall feedback")


@lru_cache(maxsize=1)
def _llm():
    return ChatGoogleGenerativeAI(model=CHAT_MODEL, temperature=0.3)


@lru_cache(maxsize=1)
def _embeddings():
    return GoogleGenerativeAIEmbeddings(model=EMBED_MODEL)


def _run_chain(prompt: ChatPromptTemplate, schema, inputs: dict, retries: int = 3):
    """prompt -> LLM -> structured output (Pydantic), retrying on temporary errors."""
    chain = prompt | _llm().with_structured_output(schema)
    last_error = None
    for attempt in range(retries):
        try:
            return chain.invoke(inputs)
        except Exception as e:  # rate limit, timeout, parse error
            last_error = e
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Model failed after {retries} attempts: {last_error}")


# ---- RAG ----
def build_retriever(job_description: str):
    """Index the job description + sample interview questions, return a retriever."""
    texts = [job_description]
    for f in DATA_DIR.glob("*.txt"):
        if f.name.startswith("sample_cv") or f.name.startswith("sample_jd"):
            continue
        texts.append(f.read_text(encoding="utf-8"))
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.create_documents(texts)
    store = FAISS.from_documents(chunks, _embeddings())
    return store.as_retriever(search_kwargs={"k": 3})


def _context(retriever, query: str) -> str:
    docs = retriever.invoke(query)
    return "\n---\n".join(d.page_content for d in docs)


# ---- Chain 1: generate questions ----
_q_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional interviewer. Write {n} interview questions "
               "tailored to this candidate and role. Mix technical and behavioral. "
               "Use the reference material for style and requirements."),
    ("human", "CV:\n{cv}\n\nJob description:\n{jd}\n\nReference material:\n{context}"),
])


def generate_questions(cv: str, jd: str, retriever, n: int = 3) -> List[str]:
    result = _run_chain(_q_prompt, QuestionList,
                        {"cv": cv, "jd": jd, "n": n, "context": _context(retriever, jd)})
    return result.questions[:n]


# ---- Chain 2: evaluate an answer ----
_e_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict but fair interviewer. Evaluate the candidate's answer "
               "to the question for the given role. Score 0-10: empty, irrelevant or "
               "'I don't know' answers must score 0-2; vague answers 3-5; solid answers "
               "6-8; excellent, specific answers with examples 9-10."),
    ("human", "Role requirements:\n{context}\n\nQuestion: {question}\n\nAnswer: {answer}"),
])


def evaluate_answer(question: str, answer: str, retriever) -> Evaluation:
    return _run_chain(_e_prompt, Evaluation, {
        "question": question,
        "answer": answer,
        "context": _context(retriever, question),
    })


# ---- Final report (no LLM needed) ----
def final_report(results: list) -> dict:
    scores = [r["evaluation"].score for r in results]
    avg = sum(scores) / len(scores) if scores else 0
    strengths = [r["question"] for r in results if r["evaluation"].score >= 7]
    return {"average": round(avg, 1), "strengths": strengths}

Writing lab.py


In [6]:
%%writefile jd_generator.py
"""Generate a job description from a target role (uses the same chain pattern as lab.py)."""
from typing import List

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

from lab import _run_chain


class JobDescription(BaseModel):
    title: str = Field(description="Job title")
    responsibilities: List[str] = Field(description="3-5 main responsibilities")
    requirements: List[str] = Field(description="5-8 key skills and requirements")
    nice_to_have: List[str] = Field(description="2-3 optional skills")


_jd_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced technical recruiter. Write a realistic, concise "
               "job description for the given role. Be specific about tools, "
               "technologies, and skills."),
    ("human", "Role: {role}"),
])


def generate_job_description(role: str) -> str:
    jd = _run_chain(_jd_prompt, JobDescription, {"role": role.strip()})
    lines = [jd.title, "", "Responsibilities:"]
    lines += [f"- {x}" for x in jd.responsibilities]
    lines += ["", "Requirements:"] + [f"- {x}" for x in jd.requirements]
    if jd.nice_to_have:
        lines += ["", "Nice to have:"] + [f"- {x}" for x in jd.nice_to_have]
    return "\n".join(lines)



class RoleAndJD(BaseModel):
    role: str = Field(description="Best-fit target job title for this candidate")
    reason: str = Field(description="One short sentence explaining why this role fits the CV")
    responsibilities: List[str] = Field(description="3-5 main responsibilities for the role")
    requirements: List[str] = Field(description="5-8 key skills and requirements")
    nice_to_have: List[str] = Field(description="2-3 optional skills")


_auto_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced technical recruiter. Read the candidate's CV and "
               "choose the single best-fit target role for them. Then write a realistic, "
               "concise job description for that role, specific about tools and skills."),
    ("human", "CV:\n{cv}"),
])


def suggest_role_and_jd(cv: str):
    """Returns (role, reason, job_description_text) using ONE LLM request."""
    r = _run_chain(_auto_prompt, RoleAndJD, {"cv": cv.strip()[:6000]})
    lines = [r.role, "", "Responsibilities:"]
    lines += [f"- {x}" for x in r.responsibilities]
    lines += ["", "Requirements:"] + [f"- {x}" for x in r.requirements]
    if r.nice_to_have:
        lines += ["", "Nice to have:"] + [f"- {x}" for x in r.nice_to_have]
    return r.role, r.reason, "\n".join(lines)

Writing jd_generator.py


In [7]:
%%writefile api.py
"""FastAPI service for the AI Interview Simulator.
Run:  uvicorn api:app --port 8000      Docs:  http://localhost:8000/docs
"""
import hashlib
import hmac
import os
from types import SimpleNamespace
from typing import List, Optional

from fastapi import Depends, FastAPI, File, HTTPException, Request, UploadFile
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, Field

from lab import build_retriever, generate_questions, evaluate_answer, final_report
from jd_generator import suggest_role_and_jd, generate_job_description


# HTTPBearer makes the "Authorize" button appear in /docs
_bearer = HTTPBearer(auto_error=False)


def require_password(
    request: Request,
    creds: Optional[HTTPAuthorizationCredentials] = Depends(_bearer),
):
    if request.url.path == "/health":      # keep health check open
        return
    expected = os.environ.get("API_PASSWORD", "")
    given = creds.credentials if creds else ""
    if not expected or not hmac.compare_digest(given.encode(), expected.encode()):
        raise HTTPException(status_code=401, detail="Wrong or missing password.")



app = FastAPI(
    title="AI Interview Simulator API",
    description="RAG + LangChain chains + Pydantic output parsing, powered by Gemini.",
    version="1.0.0",
    dependencies=[Depends(require_password)],
)

_retrievers = {}


def _retriever(job_description: str):
    key = hashlib.md5(job_description.encode("utf-8")).hexdigest()
    if key not in _retrievers:
        _retrievers[key] = build_retriever(job_description)
    return _retrievers[key]


def _fail(e: Exception):
    print("MODEL ERROR:", repr(e))
    raise HTTPException(status_code=502, detail=f"Model call failed: {e}")


# ---------- request / response schemas ----------
class CVRequest(BaseModel):
    cv: str = Field(min_length=20, description="CV text")


class RoleRequest(BaseModel):
    role: str = Field(min_length=2, examples=["AI/ML Engineer"])


class QuestionsRequest(BaseModel):
    cv: str = Field(min_length=20)
    job_description: str = Field(min_length=20)
    n: int = Field(default=3, ge=1, le=8)


class EvaluateRequest(BaseModel):
    question: str
    answer: str = Field(min_length=1)
    job_description: str = Field(min_length=20)


class ScoredAnswer(BaseModel):
    question: str
    score: int = Field(ge=0, le=10)


class ReportRequest(BaseModel):
    results: List[ScoredAnswer] = Field(min_length=1)


# ---------- endpoints ----------
@app.get("/")
def root():
    return {"message": "API is running. See /docs for endpoints."}


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/suggest-role", summary="Best-fit role + job description from a CV (1 LLM call)")
def suggest_role(req: CVRequest):
    try:
        role, reason, jd = suggest_role_and_jd(req.cv)
        return {"role": role, "reason": reason, "job_description": jd}
    except Exception as e:
        _fail(e)


@app.post("/job-description", summary="Job description from a role title (1 LLM call)")
def job_description(req: RoleRequest):
    try:
        return {"job_description": generate_job_description(req.role)}
    except Exception as e:
        _fail(e)


@app.post("/questions", summary="Generate interview questions (1 LLM call)")
def questions(req: QuestionsRequest):
    try:
        qs = generate_questions(req.cv, req.job_description,
                                _retriever(req.job_description), req.n)
        return {"questions": qs}
    except Exception as e:
        _fail(e)


@app.post("/evaluate", summary="Score one answer (1 LLM call)")
def evaluate(req: EvaluateRequest):
    try:
        ev = evaluate_answer(req.question, req.answer, _retriever(req.job_description))
        return ev.model_dump()
    except Exception as e:
        _fail(e)


@app.post("/report", summary="Final report from scores (no LLM call)")
def report(req: ReportRequest):
    items = [
        {"question": r.question, "evaluation": SimpleNamespace(score=r.score)}
        for r in req.results
    ]
    return final_report(items)

Writing api.py


In [8]:
%%writefile data/sample_questions.txt
Tell me about a challenging project you worked on and how you solved the problems you faced.
How do you handle disagreements with a teammate about a technical decision?
Explain a time you had to learn a new technology quickly. What was your approach?
How do you test and debug your code before delivering it?
Describe how you would design a simple REST API for a todo application.
What is the difference between SQL and NoSQL databases, and when would you use each?
Where do you see yourself in two years, and how does this role fit that plan?

Writing data/sample_questions.txt


In [9]:
%%writefile data/sample_cv.txt
Python developer with 2 years of experience. Built REST APIs with Django and Flask,
worked with PostgreSQL and Redis, basic Docker knowledge. Final project: a library
management system. Familiar with Git and unit testing with pytest.

Writing data/sample_cv.txt


In [10]:
%%writefile data/sample_jd.txt
Backend Engineer. Requirements: Python, REST APIs, PostgreSQL, Docker, testing,
teamwork and clear communication. Nice to have: cloud basics (AWS), CI/CD.

Writing data/sample_jd.txt


In [11]:
import os, subprocess, time, requests
from pyngrok import ngrok

try:
    ngrok.kill()
except Exception:
    pass
try:
    server.terminate()
    server.wait(timeout=5)
except Exception:
    pass

LOG = open(os.path.join(WORKDIR, "server.log"), "w")
server = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=WORKDIR, stdout=LOG, stderr=subprocess.STDOUT,
)


for i in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=1).status_code == 200:
            break
    except requests.exceptions.RequestException:
        pass
    if server.poll() is not None:   # process died
        break
    time.sleep(1)
else:
    print("Server did not start in time.")

if server.poll() is not None:
    LOG.flush()
    print("Server crashed:\n", open(os.path.join(WORKDIR, "server.log")).read())
else:
    print("Local health check:", requests.get("http://127.0.0.1:8000/health").json())

    # confirm the password protection is really on (should print 401)
    print("Without password:", requests.get("http://127.0.0.1:8000/").status_code)

    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(8000)
    PUBLIC_URL = tunnel.public_url
    print("Public URL:       ", PUBLIC_URL)
    print("Interactive docs: ", PUBLIC_URL + "/docs")

Local health check: {'status': 'ok'}
Without password: 401
Public URL:        https://giant-enduring-record.ngrok-free.dev                                     
Interactive docs:  https://giant-enduring-record.ngrok-free.dev/docs


In [12]:
import os
print(repr(os.environ["API_PASSWORD"]))
print(len(os.environ["API_PASSWORD"]))

'Q2RHdbNZG4N7IA0ofBaIn1Bg1h_YjRTh'
32
